# Ensemble Learning

---

We combine different models to create an *ensemble*. For classification the ensemle's prediction is the mode (most frequent value) of all predictions. For regression the prediction is the mean. This way we can achieve noticably higher accuracy (statistical explanation). 

There are a few ways to make an ensemble:
1. **voting**: train different algorithms on the same data (stability, balance)
2. **bagging / pasting**: train the same algorithm on differents subsets of data (reduce overfitting)
3. **boosting**: models are trained sequentially each one trained to fix the mistakes of the previous one (reduces bias)
4. **stacking**: much like voting but a model is trained on their predictions to better use their predictions (max accuracy)
5. **random subspaces** and **random patches**: 
    - random subspaces: models are trained on all instances but not all features
    - random patches: combines bagging and random subspaces (random forest base)
    
    more like part of bagging


### Voting

Sklearn class: `VotingClassifier`

hyperparameter `voting`:
- hard: uses final model predictions
- soft: uses probabilities from each model to weight more confident predictions. All models need to have `predict_proba` method. For example, for svc we need to set `probability=True` hyperparameter (under the hood this uses cv slowing the training) 

In [4]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier  # more on that below
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np

In [2]:
RANDOM_SEED = 42

In [24]:
data = fetch_covtype()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
log_clf = LogisticRegression(random_state=RANDOM_SEED)
rf_clf = RandomForestClassifier(random_state=RANDOM_SEED)
svm_clf = SVC(random_state=RANDOM_SEED)

voting_clf = VotingClassifier(
    estimators=[
        ("lr", log_clf),
        ("rf", rf_clf),
        ("svc", svm_clf),
    ],
    voting="hard",
    n_jobs=-1,  # use all cpu cores
)

for clf in [log_clf, rf_clf, svm_clf, voting_clf]:
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    print(f"{clf.__class__.__name__} accuracy: {accuracy_score(y_pred, y_test)}")

LogisticRegression accuracy: 0.7246886913418759
RandomForestClassifier accuracy: 0.9551388518368716


### Bagging & Pasting

Bagging samples data with replacement meaning the same row can be picked multiple times while others might not be picked at all (*Out-Of-Bag* data). 
Strongly reduces overfitting. Suitable even for small datasets. 

Pasting samples data without replacement meaning every row once so all rows are unique. Moderately reduces overfitting. Better fot large datasets.

In most cases *bagging is better*. 

Sklearn class: `BaggingClassifier`, `BaggingRegressor`

Hyperparameters:
- `n_estimators`: number of estimators
- `max_samples`: maximum number of samples of X used for training each base model
- **`bootstrap`: True -> bagging, False -> Pasting**
- `oob_score`: used only at bagging. Performs a validation test using the Out-Of-Bag data.

BaggingClassifier if possible uses soft voting.

In [34]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

In [4]:
d = load_breast_cancer()

X, y = d.data, d.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

In [5]:
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=RANDOM_SEED),
    n_estimators=500,
    max_samples=100,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
)

bag_clf.fit(X_train, y_train)

y_pred = bag_clf.predict(X_train)
print(f"Train accuracy: {accuracy_score(y_train, y_pred)}")

print(f"Out-Of-Bag score: {bag_clf.oob_score_}")

y_pred = bag_clf.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred)}")

Train accuracy: 0.9758241758241758
Out-Of-Bag score: 0.9472527472527472
Test accuracy: 0.956140350877193


### Random Patches & Random Subspaces

Using the `BaggingClassifier` we can use `max_features`, `bootstrap_features` hyperparameters to sample features instead of instances.

#### Random Forest

Ensemble of decision trees using random patches (bagging and feature sampling). 

Pros:
- high accuracy
- resistant to overfitting
- feature importance: shows clearly the most important features (`feature_importances_` attribute)
- missing value handling

Cons: 
- looses the explainability of the decision tree (*black-box*)
- higher cost, slower

Roughly equivalent with:
`ag_clf = BaggingClassifier(DecisionTreeClassifier(splitter="random", max_leaf_nodes=16), n_estimators=500, max_samples=1.0, bootstrap=True, n_jobs=-1)`

In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier

In [20]:
data = load_breast_cancer()

X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# no need for scaling because we use decision trees

In [23]:
rf = RandomForestClassifier(n_estimators=500, max_depth=5)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_train)
print(f"Train accuracy: {accuracy_score(y_train, y_pred)}")

y_pred = rf.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred)}")

# print("Feature: Importance")
# for feature, importance in zip(data.feature_names, rf.feature_importances_):
#     print(f"{feature}: {importance}")

Train accuracy: 0.9934065934065934
Test accuracy: 0.9649122807017544


### Boosting

Methods:
- *Adaboost* (adaptive-boosting)
- *Gradient Boosting*

#### AdaBoost

It starts by training a base model and evaluates it. Then it trains a second model focusing on the cases where the first one failed by adjusting the training set weights. This means that it trains more often on the "hard" instances. This procedure continues sequentially. The result is a weighted voting of all trained models (higher weights for better models).

Regularization: Decreasing `n_estimators` or regularizing the base estimator

#### Gradient Boosting

It starts again by training a model and evaluates it. Then it trains a second model on the residual errors of the first tree. Its prediction is the difference. This procedure continues so the error is "deleted".

It consists of `n_estimators`. We can find the optimal number of estimators using `staged_predict` because it gives predictions on each stage or simply by using early stopping mechanism.

While sklearn has `GradientBoostingClassifier` and `GradientBoostingRegressor` the best models are `XGBClassifier`, `LGBMClassifier` from XGBoost, LightGBM.

**XGBoost**: level-wise tree grow, good for small or large datasets, requires one-hot / target encoding

**LightGBM**: leaf-wise tree grow, super fast, memory light, good for large datasets (10_000+ instances), autohandles categorical data

In [36]:
from sklearn.datasets import make_classification
from sklearn.ensemble import AdaBoostClassifier

In [29]:
X, y = make_classification(n_samples=1000, n_features=20, random_state=RANDOM_SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

In [37]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=5, random_state=RANDOM_SEED),
    n_estimators=100,
    learning_rate=0.5,
)

ada.fit(X_train, y_train)

y_pred = ada.predict(X_train)
print(f"Train accuracy: {accuracy_score(y_train, y_pred)}")

y_pred = ada.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred)}")

Train accuracy: 1.0
Test accuracy: 0.91


In [70]:
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [6]:
diabetes = load_diabetes()

X, y = diabetes.data, diabetes.target
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

In [9]:
SUBSAMPLE = 0.5

gbrt = GradientBoostingRegressor(
    n_estimators=120, max_depth=3, subsample=SUBSAMPLE, random_state=RANDOM_SEED
)
gbrt.fit(X_train, y_train)

print("Original GBRT")
y_pred = gbrt.predict(X_train)
print(f"Train rmse: {np.sqrt(mean_squared_error(y_train, y_pred))}")

y_pred = gbrt.predict(X_val)
print(f"Val rmse: {np.sqrt(mean_squared_error(y_val, y_pred))}")

# find the stage with the lowest mse
errors = [mean_squared_error(y_val, y_pred) for y_pred in gbrt.staged_predict(X_val)]
best_n_estimators = np.argmin(errors) + 1

best_gbrt = GradientBoostingRegressor(
    n_estimators=best_n_estimators,
    max_depth=3,
    subsample=SUBSAMPLE,
    random_state=RANDOM_SEED,
)
best_gbrt.fit(X_train, y_train)

print(f"Optimized Estimators GBRT ({best_n_estimators} estimators)")
y_pred = best_gbrt.predict(X_train)
print(f"Train rmse: {np.sqrt(mean_squared_error(y_train, y_pred))}")

y_pred = best_gbrt.predict(X_val)
print(f"Val rmse: {np.sqrt(mean_squared_error(y_val, y_pred))}")

Original GBRT
Train rmse: 26.969492418357195
Val rmse: 57.62935950638737
Optimized Estimators GBRT (30 estimators)
Train rmse: 43.29198849968316
Val rmse: 54.872385602516836


Or simply using *early stopping* mechanism:
- `n_iter_no_change`: number of patience epochs with no validation score improvement
- `tol`: tolerance for early stoppint. We request at least tol validation score improvement

In [81]:
gbrt = GradientBoostingRegressor(
    n_estimators=120,
    max_depth=3,
    subsample=0.3,  # uses subsample percent of the data
    n_iter_no_change=10,
    validation_fraction=0.2,
    tol=1e-4,
    random_state=RANDOM_SEED,
)

gbrt.fit(X_train, y_train)

print(f"Number of estimators: {gbrt.n_estimators_}")

y_pred = gbrt.predict(X_train)
print(f"Train rmse: {np.sqrt(mean_squared_error(y_train, y_pred)):.4f}")
print(f"Train r2: {np.sqrt(r2_score(y_train, y_pred)):.4f}")

y_pred = gbrt.predict(X_val)
print(f"Val rmse: {np.sqrt(mean_squared_error(y_val, y_pred)):.4f}")
print(f"Val r2: {np.sqrt(r2_score(y_val, y_pred)):.4f}")

Number of estimators: 29
Train rmse: 45.5572
Train r2: 0.8114
Val rmse: 51.8896
Val r2: 0.7013


In [95]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

In [90]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)
# 0.25 * 0.8 = 0.2
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=RANDOM_SEED
)

In [96]:
xgbt = xgb.XGBRegressor(
    n_estimators=120,
    early_stopping_rounds=15,
    max_depth=3,
    learning_rate=0.02,
    random_state=RANDOM_SEED,
)

xgbt.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print(f"Final estimators: {xgbt.best_iteration}")

y_pred = xgbt.predict(X_train)
print(f"Train rmse: {np.sqrt(mean_squared_error(y_train, y_pred)):.4f}")
print(f"Train r2: {np.sqrt(r2_score(y_train, y_pred)):.4f}")

y_pred = xgbt.predict(X_val)
print(f"Val rmse: {np.sqrt(mean_squared_error(y_val, y_pred)):.4f}")
print(f"Val r2: {np.sqrt(r2_score(y_val, y_pred)):.4f}")

Final estimators: 109
Train rmse: 44.8550
Train r2: 0.8224
Val rmse: 54.8650
Val r2: 0.6713


We perform a quick coarse tuning

In [105]:
param_grid = {
    "max_depth": np.arange(2, 5),
    "learning_rate": np.logspace(-3, 0, num=4),
    "subsample": np.arange(0.7, 1, 0.1),
}

base_model = xgb.XGBRegressor(
    n_estimators=1000,
    early_stopping_rounds=15,
    random_state=RANDOM_SEED,
)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

best_model = grid_search.best_estimator_

print(f"Best params: {grid_search.best_params_}")

y_pred = best_model.predict(X_train)
print(f"Train rmse: {np.sqrt(mean_squared_error(y_train, y_pred)):.4f}")
print(f"Train r2: {np.sqrt(r2_score(y_train, y_pred)):.4f}")

y_pred = best_model.predict(X_val)
print(f"Val rmse: {np.sqrt(mean_squared_error(y_val, y_pred)):.4f}")
print(f"Val r2: {np.sqrt(r2_score(y_val, y_pred)):.4f}")

y_pred = best_model.predict(X_test)
print(f"Test rmse: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"Test r2: {np.sqrt(r2_score(y_test, y_pred)):.4f}")

Best params: {'learning_rate': np.float64(0.01), 'max_depth': np.int64(2), 'subsample': np.float64(0.7)}
Train rmse: 49.3152
Train r2: 0.7803
Val rmse: 52.9139
Val r2: 0.6993
Test rmse: 52.4693
Test r2: 0.6931


### Stacking

Usually it consists of 2 layers. 
- 1st layer: different **diverse** models trained on the initial data.
- 2nd layer: model trained on the predictions of the 1st layer models.

We can also use 3 layers with the 2nd layer consisting of different models trained on the predictions and the 3rd is the final predictor.

The train data is split in 2 (in general n_layers) subsets like "train_test_split" with the "test" set called *hold-out set*. The "train" subset is used to train the 1st layer. Then the first layer makes predictions on the hold-out set. The 2nd layer is then trained on these predictions.

Sklearn: `StackingClassifier`